<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# DXQuant · Derivatives Analytics with Python
## Notebook 8 · Time Grids and Business-Day Calendars

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/dxquant/blob/main/notebooks/08_dxquant_time_and_calendars.ipynb)


## Notebook Goals

This notebook streamlines the time-handling utilities used throughout DXQuant:

- convert Python `date`/`datetime` objects into sorted time grids,
- compute year fractions consistently, and
- work with a business-day calendar for scheduling and day counts.


## Setup

In Google Colab, select **Runtime → Run all** with a standard CPU runtime. This cell installs DXQuant and Matplotlib if they are missing. No GPU, Google Drive mount, or API credentials are needed. The examples use synthetic inputs.

Before the first PyPI release, installation uses the public GitHub `main` branch. When running from a local checkout, the setup cell installs the local package instead.


In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path

requirements = []
if importlib.util.find_spec("dxquant") is None:
    package_source = (
        "dxquant @ git+https://github.com/yhilpisch/dxquant.git@main"
    )
    for directory in (Path.cwd(), Path.cwd().parent):
        local_package = directory
        if (local_package / "pyproject.toml").is_file():
            package_source = str(local_package.resolve())
            break
    requirements.append(package_source)
if importlib.util.find_spec("matplotlib") is None:
    requirements.append("matplotlib>=3.8")
if requirements:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *requirements]
    )


In [ ]:
import datetime as dt

import numpy as np

import dxquant as dx

np.set_printoptions(precision=6, suppress=True)

## 1. Time Grids and Year Fractions

DXQuant normalizes mixed `date` and `datetime` inputs to a sorted `datetime` array, then computes
year fractions relative to the earliest timestamp.

In [ ]:
times = [
    dt.date(2024, 1, 3),
    dt.datetime(2024, 1, 1, 12, 0),
    dt.date(2024, 1, 2),
]

ordered = dx.ensure_datetime_array(times)
fractions = dx.year_fractions(times, day_count=365.0)

print(ordered)
print(fractions)

## 2. Uniform Grids for Monte Carlo

Monte Carlo engines typically use evenly spaced grids in year-fraction time.

In [ ]:
grid = dx.uniform_time_grid(maturity=1.0, num_steps=8)
grid

## 3. Business-Day Calendars

The `BusinessDayCalendar` wraps NumPy's business-day utilities and supports custom holidays.

In [ ]:
holidays = [dt.date(2024, 1, 1)]
cal = dx.BusinessDayCalendar(name="NYC-lite", holidays=holidays)

d0 = dt.date(2024, 1, 1)
d1 = dt.date(2024, 1, 2)
d2 = dt.date(2024, 1, 6)

print("is business day (Jan 1):", cal.is_business_day(d0))
print("is business day (Jan 2):", cal.is_business_day(d1))
print("is business day (Jan 6):", cal.is_business_day(d2))

shifted = cal.add_business_days(start=d0, offset=1)
print("Jan 1 + 1 business day:", shifted)

## 4. Business-Day Counts and Schedules

Business-day counts and schedules are often needed for coupon dates, fixing dates, and
valuation grids.

In [ ]:
start = dt.date(2024, 1, 1)
end = dt.date(2024, 1, 10)

count_excl = cal.business_days_between(start, end, inclusive=False)
count_incl = cal.business_days_between(start, end, inclusive=True)
sched = cal.schedule(start, end, include_end=True)

print("business days (exclusive):", count_excl)
print("business days (inclusive):", count_incl)
print("schedule length:", len(sched))
print("first:", sched[0], "last:", sched[-1])

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">